# Diagnostic Tests I — Solutions
### Applied Statistical Data Analysis — Prof. Dr. Kristyna Ters | MSc Finance | FHNW

---
> Complete worked solutions with short interpretations. Compare with your own attempts — the reasoning matters as much as the numbers.

In [ ]:
!pip install yfinance pandas-datareader statsmodels --quiet

import yfinance as yf
import pandas_datareader.data as web
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan, het_white, het_goldfeldquandt, acorr_breusch_godfrey
from statsmodels.stats.stattools import durbin_watson
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor':'white', 'axes.facecolor':'white',
    'axes.spines.top':False, 'axes.spines.right':False,
    'axes.grid':True, 'grid.alpha':0.3, 'font.size':11
})
YELLOW = '#FDE70E'; ORANGE = '#FCB310'; RED = '#C70101'
GREY   = '#4B4B4B'; BLUE = '#0E75FE'; GREEN = '#0B7A3C'
print('✓ Libraries loaded.')

---
# Exercise 1 — Symptom or Not? (Solution)

| # | H / A / N | Test | Reason |
|---|-----------|------|--------|
| a | **H** | Breusch-Pagan / White (GQ as opener) | the fan = variance grows with the fitted value |
| b | **A** | Durbin-Watson / Breusch-Godfrey | smooth waves = residual sticks to its own past |
| c | **H** | Breusch-Pagan / White | volatility clusters = time-varying variance |
| d | **N** | (run BP + BG anyway to confirm) | white noise is exactly what A2/A3 predict |
| e | **A** | Breusch-Godfrey (DW already flags it) | DW = 0.31 → $\hat{\rho}$ ≈ 0.85: strong positive autocorrelation, typical for levels |
| f | **N** (outlier problem) | influence diagnostics — V9 topic | neither a variance nor a correlation issue |

**Note on (f):** outliers are a *functional-form / robustness* issue — that is Diagnostic Tests II.

---
# Exercise 2 — Estimate the CAPM for a Swiss Stock

Our diagnostic patient for Exercises 2–6: the **UBS CAPM** against the SMI.

$$r_{UBSG,t} = \beta_0 + \beta_1\, r_{SMI,t} + u_t$$

In [ ]:
STOCK, INDEX = 'UBSG.SW', '^SSMI'

px  = yf.download([STOCK, INDEX], start='2020-01-01', end='2024-12-31',
                  auto_adjust=True, progress=False)['Close']
ret = px.pct_change().dropna()
# rename by label, never by position: yfinance orders the Close columns alphabetically
ret = ret.rename(columns={STOCK: 'UBS', INDEX: 'SMI'})[['UBS', 'SMI']]

X_capm = sm.add_constant(ret['SMI'])
capm   = sm.OLS(ret['UBS'], X_capm).fit()

print(f'n = {len(ret)} trading days')
print(f'beta_hat = {capm.params["SMI"]:.4f}   SE = {capm.bse["SMI"]:.4f}   '
      f't = {capm.tvalues["SMI"]:.1f}   R² = {capm.rsquared:.3f}')

**Interpretation:** a bank stock typically carries a beta above one — it is a leveraged bet on the economy. The printed SE assumes A2/A3 hold; whether we may trust it is exactly what the next exercises check.

---
# Exercise 3 — Eyes First: The Residual Plots

Produce the two standard residual plots for the UBS CAPM and read them.

**Written question:** which violation do the plots suggest — and why is that *expected* for daily return data?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(capm.fittedvalues*100, capm.resid*100, s=6, color=RED, alpha=0.4)
axes[0].axhline(0, color='black', lw=1)
axes[0].set_title('Residual vs. fitted — fan?', fontweight='bold', loc='left')
axes[0].set_xlabel('fitted (%)'); axes[0].set_ylabel('residual (%)')

axes[1].plot(ret.index, capm.resid*100, color=GREY, lw=0.6)
axes[1].axhline(0, color=RED, lw=1)
axes[1].set_title('Residual over time — volatility clusters?', fontweight='bold', loc='left')

plt.tight_layout(); plt.show()

**Reading:** the time plot shows clear **volatility clusters** (e.g. the COVID weeks and the CS-takeover turbulence of March 2023) — heteroskedasticity. That is *expected*: volatility clustering is a stylized fact of return data, so heteroskedasticity is the normal state, not the exception. The waves-type autocorrelation pattern is *absent* — returns rarely show it.

---
# Exercise 4 — Goldfeld-Quandt by Hand

Run the GQ test on the UBS CAPM, ordering by $|r_{SMI}|$ and dropping the middle ~17% of the days.

**Written question:** state $H_0$, the two df, your GQ statistic and your decision. Then name the two *choices* the test forced you to make.

In [ ]:
order = ret['SMI'].abs().sort_values().index
n     = len(order)
drop  = round(0.17 * n)
half  = (n - drop) // 2

low_idx, high_idx = order[:half], order[-half:]
m_low  = sm.OLS(ret.loc[low_idx,  'UBS'], sm.add_constant(ret.loc[low_idx,  'SMI'])).fit()
m_high = sm.OLS(ret.loc[high_idx, 'UBS'], sm.add_constant(ret.loc[high_idx, 'SMI'])).fit()

s2_low  = m_low.ssr  / m_low.df_resid
s2_high = m_high.ssr / m_high.df_resid
GQ      = s2_high / s2_low
F_crit  = stats.f.ppf(0.95, int(m_high.df_resid), int(m_low.df_resid))

print(f'n = {n}, dropped = {drop}, per half = {half}, df per half = {int(m_low.df_resid)}')
print(f'GQ = {GQ:.2f}   vs   F_crit = {F_crit:.2f}')
print('→ REJECT homoskedasticity' if GQ > F_crit else '→ do not reject')

**Answer:** $H_0$: equal error variances in the two halves ($\sigma_1^2 = \sigma_2^2$). The df are (obs per half − 2) on each side. If the GQ statistic printed above exceeds the critical value, we reject and conclude that high-|market| days carry more residual variance; if it does not, homoskedasticity survives this particular ordering and split. The two forced **choices**: the ordering variable and the size of the dropped band — both somewhat arbitrary, which is why the regression-based tests (next exercise) are the modern standard.

---
# Exercise 5 — Breusch-Pagan by Hand, then One-Liners

(a) Run the BP **auxiliary regression** yourself: regress $\hat{u}^2$ on the regressors, compute $LM = n \cdot R^2_{aux}$, and decide against $\chi^2_1$ (crit 3.84).
(b) Confirm with `het_breuschpagan` and add `het_white`.

**Written question:** why does White's test have TWO degrees of freedom here?

In [ ]:
# (a) by hand
aux = sm.OLS(capm.resid**2, X_capm).fit()
LM  = len(ret) * aux.rsquared
print(f'R²_aux = {aux.rsquared:.4f}   LM = {len(ret)} × {aux.rsquared:.4f} = {LM:.1f}   '
      f'(χ²_crit(1) = 3.84) → {"REJECT" if LM > 3.84 else "do not reject"}')

# (b) one-liners
lm_bp, p_bp, _, _ = het_breuschpagan(capm.resid, capm.model.exog)
lm_w,  p_w,  _, _ = het_white(capm.resid, capm.model.exog)
print(f'het_breuschpagan: LM = {lm_bp:.1f},  p = {p_bp:.2e}')
print(f'het_white:        LM = {lm_w:.1f},  p = {p_w:.2e}')

**Answer:** White's auxiliary regression adds the **squared** market return as a second explanatory variable, so $H_0$ now contains **two** restrictions ($a_1 = a_2 = 0$) → $\chi^2_2$, critical value 5.99. General LM rule: df = number of tested restrictions.

---
# Exercise 6 — Fix the Inference: Robust Standard Errors

Refit the UBS CAPM with **HC1** robust standard errors and compare beta, SE and t with the plain OLS fit.

**Written question:** the beta does not change at all — why exactly?

In [ ]:
capm_hc = sm.OLS(ret['UBS'], X_capm).fit(cov_type='HC1')

cmp = pd.DataFrame({
    'beta_hat': [capm.params['SMI'],  capm_hc.params['SMI']],
    'SE':       [capm.bse['SMI'],     capm_hc.bse['SMI']],
    't':        [capm.tvalues['SMI'], capm_hc.tvalues['SMI']],
}, index=['OLS', 'HC1 robust']).round(4)
print(cmp)
print(f'SE inflation: ×{capm_hc.bse["SMI"]/capm.bse["SMI"]:.3f}')

**Answer:** robust estimation changes only the **covariance formula** of the estimator, not the estimator itself — the coefficients still minimise the same sum of squared residuals. Point estimates identical, standard errors honest. In empirical finance HC standard errors are the default.

---
# Exercise 7 — A Levels Regression and Its Waves

Now the autocorrelation patient: the **mortgage pass-through in weekly levels** (FRED: `MORTGAGE30US` on `DGS10`, 2010–2024).

Estimate it, plot the residuals over time, and compute the **Durbin-Watson** statistic and the implied $\hat{\rho} = 1 - DW/2$.

In [ ]:
m30 = web.DataReader('MORTGAGE30US', 'fred', '2010-01-01', '2024-12-31')
y10 = web.DataReader('DGS10',        'fred', '2010-01-01', '2024-12-31')

lvl = m30.join(y10.resample('W-THU').mean(), how='inner').dropna()
# rename by label, never by position
lvl = lvl.rename(columns={'MORTGAGE30US': 'mort', 'DGS10': 'y10'})[['mort', 'y10']]

X_mort = sm.add_constant(lvl['y10'])
mort   = sm.OLS(lvl['mort'], X_mort).fit()

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(lvl.index, mort.resid, color=GREY, lw=1.1)
ax.axhline(0, color=RED, lw=1)
ax.set_title('Mortgage-level regression: residuals over time', fontweight='bold', loc='left')
plt.tight_layout(); plt.show()

dw = durbin_watson(mort.resid)
print(f'n = {len(lvl)} weeks   DW = {dw:.2f}   implied rho_hat = 1 − DW/2 = {1 - dw/2:.2f}')

**Reading:** smooth waves instead of noise, and DW near zero → $\hat{\rho}$ near one: the residuals are almost a random walk. Typical for regressions in **levels** — the printed OLS standard errors are far too optimistic.

---
# Exercise 8 — Breusch-Godfrey: Test the Memory Formally

Run BG with $p = 4$ lags on the levels regression — once **by hand** (auxiliary regression) and once with `acorr_breusch_godfrey`.

**Written question:** why 4 lags — and what are the df of the $\chi^2$?

In [ ]:
p = 4
u = mort.resid
aux_df = pd.DataFrame({'y10': lvl['y10']})
for j in range(1, p+1):
    aux_df[f'u_lag{j}'] = u.shift(j)
aux_df['u'] = u
aux_df = aux_df.dropna()

aux_bg = sm.OLS(aux_df['u'], sm.add_constant(aux_df.drop(columns='u'))).fit()
LM_bg  = len(aux_df) * aux_bg.rsquared
print(f'By hand:  R²_aux = {aux_bg.rsquared:.3f}   LM = {len(aux_df)} × {aux_bg.rsquared:.3f} = {LM_bg:.1f}'
      f'   (χ²_crit(4) = {stats.chi2.ppf(0.95, 4):.2f})')

# statsmodels is changing what these tests return: from release 0.16 on,
# acorr_breusch_godfrey and het_goldfeldquandt hand back a named result object
# instead of a plain tuple. Positional INDEXING works in both worlds, fixed-arity
# unpacking does not, so read element 0 (statistic) and element 1 (p-value).
bg = acorr_breusch_godfrey(mort, nlags=4)
lm_bg, p_bg = bg[0], bg[1]
print(f'One line: LM = {lm_bg:.1f},  p = {p_bg:.2e}')

**Answer:** $p$ is the analyst's choice — a convention, not a formula. Four lags cover roughly a month of error memory in weekly data; with autocorrelation this strong, any reasonable $p$ rejects. The $\chi^2$ **df equal $p$** — the number of tested restrictions $\rho_1 = \dots = \rho_4 = 0$.

---
# Exercise 9 — Newey-West: Honest Standard Errors

(a) Compute the rule-of-thumb bandwidth $L \approx 0.75\,n^{1/3}$ for the mortgage sample.
(b) Refit with `cov_type='HAC'` and compare SE and t with plain OLS.

**Written question:** your sample has ~750 weeks. Roughly how large would $n$ have to be for the rule of thumb to allow **twice** as many lags?

In [ ]:
n_m = len(lvl)
L   = int(np.ceil(0.75 * n_m**(1/3)))
print(f'L ≈ 0.75 × {n_m}^(1/3) = {0.75 * n_m**(1/3):.1f}  →  L = {L}')

mort_nw = sm.OLS(lvl['mort'], X_mort).fit(cov_type='HAC', cov_kwds={'maxlags': L})

cmp = pd.DataFrame({
    'beta_hat': [mort.params['y10'],  mort_nw.params['y10']],
    'SE':       [mort.bse['y10'],     mort_nw.bse['y10']],
    't':        [mort.tvalues['y10'], mort_nw.tvalues['y10']],
}, index=['OLS', f'Newey-West (L={L})']).round(4)
print(cmp)

**Answer:** $L$ grows with the *cube root* of $n$: doubling the lags needs $2^3 = 8$ times the data — roughly **6000 weeks** (over a century!). The bandwidth grows very slowly, which is the point: high-order autocovariances are too noisy to be worth much. And remember: $L$ tunes the covariance *estimator* — it is not a statement about the true model.

---
# Exercise 10 — The Better Fix: Dynamics + Information Criteria

(a) Estimate the dynamic model $mort_t = \alpha + \sum_{j=1}^{p} \varphi_j\, mort_{t-j} + \theta\, y10_t + u_t$ for $p = 0, \dots, 4$ and tabulate **AIC and BIC** (Brooks' formulas: $\ln(\hat{\sigma}^2) + 2k/T$ and $\ln(\hat{\sigma}^2) + (k/T)\ln T$).
(b) Pick $p$ by the BIC minimum, re-estimate, and report: the adjustment coefficient, the short-run effect $\theta$, and the **long-run pass-through** $\theta / (1 - \sum\varphi_j)$.
(c) Re-run Breusch-Godfrey on the chosen model.

**Written question:** why must the final autocorrelation check use BG and **not** Durbin-Watson?

In [ ]:
P_MAX = 4
# AIC and BIC are only comparable across models estimated on the SAME sample. Every
# extra lag would otherwise cost one observation at the start of the sample, which
# lowers RSS for mechanical reasons. Fix the estimation window at the first date the
# largest model (p = P_MAX) can use.
FIRST = lvl.index[P_MAX]

def dynamic_model(p):
    df = pd.DataFrame({'y10': lvl['y10']})
    for j in range(1, p+1):
        df[f'mort_lag{j}'] = lvl['mort'].shift(j)
    df['mort'] = lvl['mort']
    df = df.dropna().loc[FIRST:]          # identical sample for every p
    X = sm.add_constant(df.drop(columns='mort'))
    return sm.OLS(df['mort'], X).fit(), df

rows = []
for p in range(P_MAX + 1):
    m, df_p = dynamic_model(p)
    T, k = len(df_p), int(m.df_model) + 1
    sig2 = m.ssr / T
    rows.append({'lags p': p,
                 'AIC': np.log(sig2) + 2*k/T,
                 'BIC': np.log(sig2) + k/T*np.log(T)})
ic = pd.DataFrame(rows).set_index('lags p').round(4)
print(ic)

p_star = int(ic['BIC'].idxmin())
m_dyn, _ = dynamic_model(p_star)
phi   = m_dyn.params.filter(like='mort_lag').sum()
theta = m_dyn.params['y10']
print(f'\nBIC picks p = {p_star}')
print(f'Adjustment: {phi:.3f}   short-run θ: {theta:.3f}   '
      f'long-run pass-through: {theta/(1-phi):.3f}')

bg_dyn = acorr_breusch_godfrey(m_dyn, nlags=4)   # indexed, not unpacked
lm_dyn, p_dyn = bg_dyn[0], bg_dyn[1]
print(f'\nBG on the dynamic model: LM = {lm_dyn:.1f}, p = {p_dyn:.3f}'
      + ('  → clean' if p_dyn > 0.05 else '  → still autocorrelated'))

**Answer:** with a **lagged dependent variable** among the regressors, Durbin-Watson is invalid — it is biased towards 2 and will look healthy even when it should not. Breusch-Godfrey remains valid in exactly this situation, which is one of the two reasons it beats DW.

**The moral of the chapter:** Newey-West repaired the *inference*; the lag repaired the *model* — and the information criteria told us how many lags the model deserves. (They return in force when we choose ARMA orders later in the course.)

---
*Applied Statistical Data Analysis | Prof. Dr. Kristyna Ters | FHNW School of Business | HS 2026*